# DFU Results Viewer

This notebook loads training metrics and saved figures from the `outputs/` directory so you can inspect Stage 1, Stage 2, ablation, and XAI results after a run.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

OUTPUT_DIR = Path('outputs')
print(f'Using output directory: {OUTPUT_DIR.resolve()}')
print('Files found:', len(list(OUTPUT_DIR.glob('*'))))

In [ ]:
def load_json(path):
    if not path.exists():
        print(f'Missing: {path.name}')
        return None
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)

stage1 = load_json(OUTPUT_DIR / 'stage1_history.json')
stage2 = load_json(OUTPUT_DIR / 'stage2_best_metrics.json')
ablation = load_json(OUTPUT_DIR / 'ablation_results.json')

if stage1:
    stage1_summary = pd.DataFrame([
        {
            'best_val_dice': stage1.get('best_val_dice'),
            'final_train_loss': stage1['history']['train_loss'][-1] if stage1['history']['train_loss'] else None,
            'num_epochs_recorded': len(stage1['history']['epoch'])
        }
    ])
    display(stage1_summary)

if stage2:
    stage2_summary = pd.DataFrame([
        {
            'best_epoch': stage2.get('epoch'),
            'val_f1_weighted': stage2.get('val_f1_weighted'),
            'val_f1_macro': stage2.get('val_f1_macro'),
            'val_acc': stage2.get('val_acc')
        }
    ])
    display(stage2_summary)
    
    per_class = pd.DataFrame({
        'precision': stage2.get('per_class_precision', []),
        'recall': stage2.get('per_class_recall', []),
        'f1': stage2.get('per_class_f1', [])
    }, index=[f'Grade {i}' for i in range(1, len(stage2.get('per_class_f1', [])) + 1)])
    display(per_class)

if ablation:
    ablation_df = pd.DataFrame.from_dict(ablation, orient='index')
    display(ablation_df)

In [ ]:
figure_names = [
    'stage1_training_curves.png',
    'stage2_loss_curves.png',
    'stage2_f1_curves.png',
    'stage2_confusion_matrix.png'
]

for figure_name in figure_names:
    figure_path = OUTPUT_DIR / figure_name
    if figure_path.exists():
        print(f'Displaying {figure_name}')
        display(Image.open(figure_path))
    else:
        print(f'Missing: {figure_name}')

In [ ]:
xai_dir = OUTPUT_DIR / 'xai'
if xai_dir.exists():
    xai_images = sorted(xai_dir.glob('xai_sample_*.png'))
    print(f'XAI images found: {len(xai_images)}')
    for image_path in xai_images[:5]:
        print(f'Displaying {image_path.name}')
        display(Image.open(image_path))
else:
    print('No XAI directory found yet.')